# 00 — Smoke test device (pilier État)

Objectif : **valider l'environnement DL avant toute modélisation**.

1. **Check machine** (sans torch) → recommande l'env PyTorch à installer.
2. Versions torch/torchvision + device retenu (`get_device`).
3. Tenseur + convolution sur le device (ça calcule vraiment ?).
4. Mini-chrono CPU vs device (l'accélération est réelle ?).

Bloquant : ne pas passer au Stage 1 si le device attendu (MPS sur ce Mac) n'est pas exploitable.

In [1]:
import sys
sys.path.insert(0, "../src")
import importlib, device; importlib.reload(device)
from device import get_device, describe_machine, recommend_install

# --- Check machine : tourne SANS torch, sert à choisir le bon env d'install ---
for k, v in describe_machine().items():
    print(f"{k:10s}: {v}")
print("\nRecommandation install :\n ", recommend_install())

system    : Darwin
machine   : arm64
processor : arm
python    : 3.12.12

Recommandation install :
  Mac Apple Silicon → PyTorch par défaut (wheels PyPI, backend MPS). Ex : `uv add torch torchvision` (device attendu : mps).


In [2]:
# --- Versions + device retenu (nécessite torch installé) ---
import torch, torchvision
print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
print("mps dispo  :", torch.backends.mps.is_available())
print("cuda dispo :", torch.cuda.is_available())

device = get_device()
print("device     :", device)

torch      : 2.13.0
torchvision: 0.28.0
mps dispo  : True
cuda dispo : False
device     : mps


In [3]:
# --- Tenseur + convolution sur le device : ça calcule vraiment ? ---
x = torch.randn(8, 3, 64, 64, device=device)
conv = torch.nn.Conv2d(3, 16, kernel_size=3, padding=1).to(device)
y = conv(x)
print("entrée :", tuple(x.shape), "->  sortie :", tuple(y.shape))
print("sortie sur :", y.device, "| moyenne :", float(y.mean()))

entrée : (8, 3, 64, 64) ->  sortie : (8, 16, 64, 64)
sortie sur : mps:0 | moyenne : 0.013909552246332169


/var/folders/3l/v1pvxhld5873jqqxvwq3r_w00000gn/T/ipykernel_17399/906477122.py:6: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:823.)
  print("sortie sur :", y.device, "| moyenne :", float(y.mean()))


In [4]:
# --- Mini-chrono CPU vs device (matmul 4096x4096) ---
import time

def bench(dev, n=4096, iters=5):
    a = torch.randn(n, n, device=dev)
    b = torch.randn(n, n, device=dev)
    if dev.type in ("mps", "cuda"):
        (a @ b).sum().item()  # warmup + sync
    t0 = time.perf_counter()
    for _ in range(iters):
        c = a @ b
    c.sum().item()  # force la synchro avant de mesurer
    return (time.perf_counter() - t0) / iters

t_cpu = bench(torch.device("cpu"))
print(f"CPU    : {t_cpu*1e3:7.1f} ms / matmul")
if device.type != "cpu":
    t_dev = bench(device)
    print(f"{device.type:6s} : {t_dev*1e3:7.1f} ms / matmul  ->  x{t_cpu/t_dev:.1f} plus rapide")

CPU    :    86.5 ms / matmul


mps    :    31.7 ms / matmul  ->  x2.7 plus rapide


**Verdict attendu (Mac M1 Pro)** : `mps dispo = True`, device `mps`, la convolution sort une
shape `(8, 16, 64, 64)`, et le matmul est plus rapide sur MPS que sur CPU. Si `mps dispo = False`,
régler l'install (cf. recommandation cellule 1) **avant** d'attaquer le Stage 1.